## Today {.smaller}

A large language model is the decoder from last week, trained on enough text
that the next token is often a useful answer, then adjusted so it answers the
question instead of continuing the prompt.

Today is the practical half. Position had to change so a long prompt still
works. After that, the work is running the finished weights. The three jobs
are embeddings, a small fine-tune, and hosting.

## What an LLM is

A large language model reads the tokens so far and returns a probability for
every token in the vocabulary. You draw one token, append it, and repeat.

The block is the one from last week: attention, a feed-forward layer, a
residual, and a normalization. What changed is the size, the data, and two
later training runs that change how it behaves.

## Three training runs {.smaller}

[![](images/fig-stages.png){.dw92}](images/fig-stages.png){target="_blank" .zoom}

RLHF learns a reward from human rankings of two answers, then trains the model
against that reward. DPO skips the reward model and trains on the pairs. A
preference run cannot add a fact that was never in the pretraining text.

## Scale {.smaller}

[![](images/fig-growth.png){.dw80}](images/fig-growth.png){target="_blank" .zoom}

The points are public models through 2023. The architecture on the left is
last week's. An 8 billion parameter model is something you can run. A frontier
model is something you call. You do not train either one for a class project,
or for most jobs.

## Absolute position

Last week, position was a vector added to the word. The model learned a slot:
this word, at this index.

A model trained on sequences of length 2,048 has no vector for index 8,000.
Handing it a longer prompt does not work, because that slot was never trained.

## Rotary position embedding {.smaller}

Rotary position embedding stores the position as a rotation. The same word is
the same vector, turned by its position times one base angle.

[![](images/rope-rotation.png){.dw52}](images/rope-rotation.png){target="_blank" .zoom}

The query and the key are both rotated, so the attention score depends on the
gap between them. A gap the model has seen still works further along in a
longer prompt, which is why the context window can extend past the training
length.

## Three later replacements {.smaller}

Three names are enough to read a current model card.

**RMSNorm** is the normalization current models use. It rescales a vector and
does not subtract the mean. It replaced LayerNorm because it is cheaper at
this size.

**Grouped-query attention** shares one set of keys and values across several
query heads. Fewer keys and values means a smaller cache, which is what makes
a long context fit in memory.

**SwiGLU** is the activation inside the feed-forward layer. It replaced the
two-layer network from last week.

## Model size and memory {.smaller}

The number in the name, 8B or 70B, is the parameter count. Whether it runs on
your machine is the memory.

At 16-bit precision each parameter is 2 bytes, so the weights of an 8 billion
parameter model are about 16 GB. **Quantization** stores each weight in fewer
bits. At 4 bits those weights are about 4 GB, which is small enough to load on
a laptop. The network is the same shape. It is a little less accurate.

The context window is counted in tokens. In English a token is roughly three
quarters of a word, so a 4,096-token window is about 3,000 words.

## Where you run it {.smaller}

[![](images/fig-where.png){.dw92}](images/fig-where.png){target="_blank" .zoom}

Hugging Face is the path when you need the tensors: the next-token
probabilities, an embedding, or a fine-tune. Ollama and LM Studio return an
answer.

## Temperature and top-p {.smaller}

The model returns a distribution. One completion is one draw from it.

[![](images/fig-decoding.png){.dw92}](images/fig-decoding.png){target="_blank" .zoom}

**Temperature** divides the scores before the softmax. Low temperature makes
the top token nearly certain. High temperature flattens the distribution, so
unlikely tokens get drawn. **Top-p** keeps the smallest set of tokens whose
probabilities add up to $p$, and samples inside that set.

Log the prompt, fix the temperature, and read several samples before you
decide the model is wrong. A bad sentence can be the draw.

## Prefill and decode {.smaller}

[![](images/fig-serve.png){.dw92}](images/fig-serve.png){target="_blank" .zoom}

Reading the prompt is one forward pass. Writing the answer is a new forward
pass for every token, and each pass reads the weights. A long answer costs
more than a long prompt. An API bill follows the tokens the model writes.

## KV cache {.smaller}

The KV cache stores the keys and values of tokens already seen. The next token
reads them instead of recomputing the prefix.

[![](images/fig-kvcache.png){.dw85}](images/fig-kvcache.png){target="_blank" .zoom}

The time for one more token stays flat once the past is cached. The cost moves
to memory: the cache grows with the conversation. A long chat can run out of
GPU memory even when the weights themselves fit.

## Embeddings {.smaller}

An embedding model is a transformer that turns a piece of text into one vector.
Search compares the question's vector with vectors already stored.

[![](images/fig-embed.png){.dw95}](images/fig-embed.png){target="_blank" .zoom}

## Ingestion {.smaller}

Ingestion is preparing text so it can be searched later. You split a document
into chunks, embed each chunk with that same model, and store the vectors.

[![](images/fig-ingest.png){.dw92}](images/fig-ingest.png){target="_blank" .zoom}

## Parameter-efficient fine-tuning {.smaller}

Parameter-efficient fine-tuning (PEFT) trains a small correction and leaves
the pretrained weights fixed. LoRA is the usual version.

[![](images/fig-lora.png){.dw92}](images/fig-lora.png){target="_blank" .zoom}

Use it when the failure is behavior: a format, a tone, a domain vocabulary.
A missing fact belongs in the prompt, or in retrieval.

## Hosting {.smaller}

Hosting is the decision of where the weights sit and who pays for each
generated token.

A local model keeps the documents on the machine, and it is limited to what
fits in memory. An API model is stronger, and the prompt leaves. A server you
run is the case in between: one GPU, the documents stay, and the limit is how
many conversations the cache can hold at once.

The expensive half is decode. A design that writes a long answer on every
request costs more than one that retrieves a short passage and writes a short
answer.

## Agents {.smaller}

An agent is a language model plus an orchestrator. The orchestrator is a
program outside the model. It reads the text the model writes and runs a
tool only when that text matches a tool call. The format of the call is
fixed in advance.

[![](images/fig-agent.png){.dw100}](images/fig-agent.png){target="_blank" .zoom}

## Chain of thought {.smaller}

Chain of thought is a prompt that asks the model to write the steps before the
final answer. The weights stay the same. Each written step is another forward
pass, so a hard question gets more computation.

[![](images/chain-of-thought.png){.dw90}](images/chain-of-thought.png){target="_blank" .zoom}

The table is Kojima et al. (2022), one model, text-davinci-002. The paired
numbers are two ways of reading the final answer out of the generated text.
The row is the prompt: the second row adds "Let's think step by step." On
MultiArith, accuracy goes from about 18 to about 79. On a one-step question
it barely moves. An agent uses the same habit: write the step, then call the
tool.

## Checking a sample {.smaller}

With temperature above zero, two runs of the same prompt can differ, and a
failure can be the draw. One run does not settle a bug report.

What you can do is small and specific. Fix the temperature. Save the prompt
and several samples. If the bad answer shows up in every draw, the problem is
the model or the prompt. If it shows up once, it is the sample.

The same habit applies to an agent. Log the tool calls, not only the final
paragraph. The wrong step is usually in the loop, and the final paragraph
hides it.

## Wrap-up {.smaller}

- An LLM is last week's decoder, at a size you download.
- Pretraining adds the knowledge. The later runs change the behavior.
- RoPE stores position as a rotation, which is why a long context still works.
- 8B at 16-bit is about 16 GB. At 4-bit it is about 4 GB and fits on a laptop.
- Embed chunks to store text, and embed the question to search it. LoRA when
  the failure is behavior. Retrieve
  when the failure is a missing fact.
- Decode, one token at a time, is the expensive half. The cache is why a long
  chat costs memory.
- An agent is that model plus an orchestrator, which runs a tool when the
  model writes a tool call.

## Supplemental content {.smaller}

Optional. Prof. Hickman's [centralized lecture
content](https://jfh.georgetown.domains/centralized-lecture-content/):

- [Large language models](https://jfh.georgetown.domains/centralized-lecture-content/content/machine-learning/deep-learning/transformers/llm/notes.html),
  the longer notes behind today, including the RoPE figure
- [Running an LLM locally](https://jfh.georgetown.domains/centralized-lecture-content/content/machine-learning/deep-learning/transformers/running-and-LLM-locally/notes.html):
  Ollama, LM Studio, and Hugging Face
- [Agentic AI](https://jfh.georgetown.domains/centralized-lecture-content/content/machine-learning/deep-learning/transformers/agentic-ai/notes.html):
  the loop, tools, and chain of thought

Also optional: Kojima et al., [Large Language Models are Zero-Shot
Reasoners](https://arxiv.org/abs/2205.11916) (2022), the chain-of-thought
table; Hu et al., [LoRA](https://arxiv.org/abs/2106.09685) (2021); Su et al.,
[RoFormer](https://arxiv.org/abs/2104.09864) (2021), the rotary embedding.

## Lab 9 and Quiz 9 {.smaller}

**Lab 9** is running DistilGPT-2 locally. You read the next-token distribution,
sample at two temperatures, embed chunks and rank a question against them,
and run a tool only when the text contains the call. There is no training
run. It is due Wednesday, November 4.

**Quiz 9** is now. Scope is the Oct 22 lecture and Lab 8: attention, query,
key and value, multi-head, positional encoding, the masks, the block, and
next-word generation. A formula sheet comes with it.

Next week is **Generative Modeling**: autoencoders, latent spaces, and VAEs.